## tl;dr

개방ID `4293097138`은 2009~2025년 강원 춘천시 대학원으로 관측된다. 2025년 9개 패널·102행의 실질 측정값은 모두 0이고, 일본학과·지역연구학과·컨벤션전시학과는 모두 `폐과`다. 2009·2013·2016년에는 각각 졸업생 1·2·1명이 기록되며, 세 해 모두 KEDI의 `한림대학교국제학대학원`(폐교)이 동일 후보로 나타난다.

## Context & Methods

2025년 전체 패널의 ID 존재 여부와 비영 측정값, 0101의 연도별 핵심 지표, 1017의 학과 상태, KEDI 학교명 후보 반복 여부를 확인한다.

### Key Assumptions

- 개방ID는 앞자리 보존을 위해 문자열로 처리한다.
- 0은 결측치와 구분된 명시적 값이다.
- KEDI 후보는 학교명 확정 매핑이 아니라 수치·지역·학교유형이 일치한 검토 단서다.

## Data

In [1]:
import csv
import decimal
import os
from pathlib import Path
import duckdb

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
database_path = Path(os.environ.get(
    'EDSS_DUCKDB_PATH',
    '/Users/joocheol/Documents/GitHub/edss/data/processed/edss/restricted/edss_all.duckdb',
))
candidate_path = repo_root / 'data/processed/edss_0101_kedi_row_match_evidence_2009_2025.csv'
assert database_path.exists()
assert candidate_path.exists()
connection = duckdb.connect(str(database_path), read_only=True)
open_id = '4293097138'
database_path.name, candidate_path.name, duckdb.__version__

('edss_all.duckdb', 'edss_0101_kedi_row_match_evidence_2009_2025.csv', '1.4.1')

## Results

### 1. 2025년 패널별 행 수와 비영 측정값 검사

In [2]:
table_names = [
    ('higher_education', 'panel_0101'),
    ('higher_education', 'panel_0104'),
    ('higher_education', 'panel_0105'),
    ('higher_education', 'panel_0231'),
    ('higher_education', 'panel_0246'),
    ('university_disclosure', 'panel_0306'),
    ('university_disclosure', 'panel_0308'),
    ('university_disclosure', 'panel_0715'),
    ('university_disclosure', 'panel_1017'),
]
dimension_columns = {
    '조사년도', '개방ID', '적용년도', '학기구분명', '수업연한명', '개설기간명',
    '학년명', '성별명', '학위과정구분명', '학과명', '학과상태명', '단과대학명',
    '교육부계열명', '학과계열구분명', '주야간계절구분명', '본분교명', '시도명',
    '지역명', '학교구분명', '학제유형명',
}
panel_rows = []
nonzero_measure_cells = []
for schema_name, table_name in table_names:
    cursor = connection.execute(
        f"SELECT * FROM {schema_name}.{table_name} WHERE 개방ID = ? AND 조사년도 = '2025'",
        [open_id],
    )
    rows = cursor.fetchall()
    columns = [item[0] for item in cursor.description]
    panel_rows.append((schema_name, table_name, len(rows)))
    for column_index, column_name in enumerate(columns):
        if column_name.startswith('_') or column_name in dimension_columns:
            continue
        for row in rows:
            value = row[column_index]
            try:
                numeric_value = decimal.Decimal(str(value).replace(',', ''))
            except decimal.InvalidOperation:
                continue
            if numeric_value != 0:
                nonzero_measure_cells.append((schema_name, table_name, column_name, value))

assert len(panel_rows) == 9
assert sum(item[2] for item in panel_rows) == 102
assert nonzero_measure_cells == []
panel_rows, nonzero_measure_cells

([('higher_education', 'panel_0101', 1),
  ('higher_education', 'panel_0104', 1),
  ('higher_education', 'panel_0105', 2),
  ('higher_education', 'panel_0231', 6),
  ('higher_education', 'panel_0246', 12),
  ('university_disclosure', 'panel_0306', 3),
  ('university_disclosure', 'panel_0308', 5),
  ('university_disclosure', 'panel_0715', 54),
  ('university_disclosure', 'panel_1017', 18)],
 [])

### 2. 0101 연도별 핵심 지표

In [3]:
yearly_sql = '''
SELECT 조사년도, 지역명, 고등교육학교_재적학생수, 고등교육학교_학과수,
       고등교육학교_교원수, 고등교육학교_졸업생수
FROM higher_education.panel_0101
WHERE 개방ID = '4293097138'
ORDER BY 조사년도
'''
yearly_rows = connection.execute(yearly_sql).fetchall()
assert len(yearly_rows) == 17
[(row[0], row[5]) for row in yearly_rows if row[5] != '0'], yearly_rows

([('2009', '1'), ('2013', '2'), ('2016', '1')],
 [('2009', '강원 춘천시', '0', '0', '0', '1'),
  ('2010', '강원 춘천시', '0', '0', '0', '0'),
  ('2011', '강원 춘천시', '0', '0', '0', '0'),
  ('2012', '강원 춘천시', '0', '0', '0', '0'),
  ('2013', '강원 춘천시', '0', '0', '0', '2'),
  ('2014', '강원 춘천시', '0', '0', '0', '0'),
  ('2015', '강원 춘천시', '0', '0', '0', '0'),
  ('2016', '강원 춘천시', '0', '0', '0', '1'),
  ('2017', '강원 춘천시', '0', '0', '0', '0'),
  ('2018', '강원 춘천시', '0', '0', '0', '0'),
  ('2019', '강원 춘천시', '0', '0', '0', '0'),
  ('2020', '강원 춘천시', '0', '0', '0', '0'),
  ('2021', '강원 춘천시', '0', '0', '0', '0'),
  ('2022', '강원 춘천시', '0', '0', '0', '0'),
  ('2023', '강원 춘천시', '0', '0', '0', '0'),
  ('2024', '강원 춘천시', '0', '0', '0', '0'),
  ('2025', '강원 춘천시', '0', '0', '0', '0')])

### 3. 학과와 폐과 상태

In [4]:
department_sql = '''
SELECT 조사년도, 학과상태명,
       STRING_AGG(DISTINCT 학과명, ' | ' ORDER BY 학과명) AS 학과명,
       COUNT(*) AS 행수
FROM university_disclosure.panel_1017
WHERE 개방ID = '4293097138'
GROUP BY 조사년도, 학과상태명
ORDER BY 조사년도, 학과상태명
'''
department_rows = connection.execute(department_sql).fetchall()
assert {row[1] for row in department_rows} == {'폐과'}
department_rows

[('2009', '폐과', '일본학과 | 컨벤션전시학과', 18),
 ('2011', '폐과', '일본학과 | 컨벤션전시학과', 12),
 ('2012', '폐과', '일본학과 | 컨벤션전시학과', 18),
 ('2013', '폐과', '일본학과 | 컨벤션전시학과', 12),
 ('2015', '폐과', '일본학과 | 컨벤션전시학과', 6),
 ('2016', '폐과', '일본학과 | 지역연구학과 | 컨벤션전시학과', 18),
 ('2017', '폐과', '일본학과 | 지역연구학과 | 컨벤션전시학과', 18),
 ('2018', '폐과', '일본학과 | 지역연구학과 | 컨벤션전시학과', 18),
 ('2019', '폐과', '일본학과 | 지역연구학과 | 컨벤션전시학과', 18),
 ('2020', '폐과', '일본학과 | 지역연구학과 | 컨벤션전시학과', 18),
 ('2021', '폐과', '일본학과 | 지역연구학과 | 컨벤션전시학과', 18),
 ('2022', '폐과', '일본학과 | 지역연구학과 | 컨벤션전시학과', 18),
 ('2023', '폐과', '일본학과 | 지역연구학과 | 컨벤션전시학과', 18),
 ('2024', '폐과', '일본학과 | 지역연구학과 | 컨벤션전시학과', 18),
 ('2025', '폐과', '일본학과 | 지역연구학과 | 컨벤션전시학과', 18)]

### 4. KEDI 학교명 후보 반복 확인

In [5]:
with candidate_path.open(encoding='utf-8-sig', newline='') as handle:
    candidate_rows = [row for row in csv.DictReader(handle) if row['openid'] == open_id]
candidate_preview = [
    {key: row[key] for key in ['year', 'kedi_school_name', 'kedi_school_status', 'kedi_region', 'match_families']}
    for row in candidate_rows
]
assert len(candidate_rows) == 3
assert {row['kedi_school_name'] for row in candidate_rows} == {'한림대학교국제학대학원'}
assert {row['kedi_school_status'] for row in candidate_rows} == {'폐교'}
candidate_preview

[{'year': '2009',
  'kedi_school_name': '한림대학교국제학대학원',
  'kedi_school_status': '폐교',
  'kedi_region': '강원 춘천시',
  'match_families': 'graduates'},
 {'year': '2013',
  'kedi_school_name': '한림대학교국제학대학원',
  'kedi_school_status': '폐교',
  'kedi_region': '강원 춘천시',
  'match_families': 'graduates'},
 {'year': '2016',
  'kedi_school_name': '한림대학교국제학대학원',
  'kedi_school_status': '폐교',
  'kedi_region': '강원 춘천시',
  'match_families': 'graduates'}]

## Takeaways

- 2025년에는 9개 패널·102행이 있으나 모든 실질 측정값이 0이다.
- 일본학과·지역연구학과·컨벤션전시학과는 2025년을 포함한 관측 연도에서 모두 폐과로 표시된다.
- 0101에서 2009년 1명, 2013년 2명, 2016년 1명의 졸업생만 예외적으로 남고, 동일한 세 해에 `한림대학교국제학대학원`(폐교)이 KEDI 후보로 반복된다.
- 기관 후보의 개연성은 높지만 직접 학교명 연결은 0회이므로 자동 조인하지 않고 수동 후보로 유지한다.